# Basic Prompting

*DIY = Do it yourself

### Setup

#### Load the API key and relevant Python libraries


-  Google Drive Mount 하기.

(참고: 구글 계정 로그인 필요하며, 접근 권한을 승인해야합니다. 자동으로 구글 보안 알림 메일이 전송됩니다)


In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


-  Working Directory 수정하기 (작업 폴더 `2025_SNUIPS_Workshop`)

In [ ]:
import os
os.chdir("/content/drive/MyDrive/2025_SNUIPS_Workshop/")
print(os.getcwd())

/content/drive/MyDrive/2025_SNUIPS_Workshop


- .env파일 안에 저장한 openai API key 불러오기

API Key는 절대 타인에게 공유하면 안됩니다. 본 워크샵에서 사용할 api key는 워크샵이 종료되면 삭제되며 이후로는 사용할 수 없습니다. 구글 코랩 노트북에 직접 프린팅하는 것도 지양해야합니다.

안전한 사용을 위해 .env파일을 만들고 그 안에 key를 저장합니다. 이후로는 직접 api key를 입력하거나 정의할 필요 없이, 이 .env파일만 불러오면 됩니다.

In [ ]:
# Install the python-dotenv library. This library allows you to load environment variables from a .env file.
# Environment variables are a common way to store sensitive information like API keys, database credentials, etc.
# Separating these credentials from your code makes it more secure.
!pip install python-dotenv -q
!pip install openai==0.28 -q #downgrading openai version

import openai
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

openai.api_key  = os.getenv('OPENAI_API_KEY')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 2.3 MB/s eta 0:00:00


- 현재 openai version 확인하기

In [ ]:
# check OpenAI version
openai.__version__

'0.28.0'

- OpenAI의 ChatCompletion function 정의하기

- [OpenAI ChatCompletion](https://platform.openai.com/docs/api-reference/chat)

- [Model Specifications: GPT and reasoning models](https://platform.openai.com/docs/models#gpt-models)

openai==0.28버전인지 확인하기 (아닐 경우 런타임 > 세션 다시 시작하고 다시 run)

In [ ]:
## we'll degrade the openai version to use the gpt-3.5-turbo model for now
## you may need to restart the session after the installation & running this code
#!pip install openai==0.28 -q

def get_completion(prompt, model="gpt-3.5-turbo"):
    messages = [{"role": "user", "content": prompt}]
    response = openai.ChatCompletion.create(
        model=model,
        messages=messages,
        temperature=0, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message["content"]

# Prompting Principles

- Principle 1: Write clear and specific instructions.
(I highly recommend using English, as Korean grammar often omits subjects or objects, especially in casual language. Additionally, translating English terms into Korean may confuse the model.)

- Principle 2: Give the model time to "think."

# Use cases

### Task 1: Use deliminaters to clearly indicate distinct parts of the input

Delimiters can be anything like
```
``` ```, """, <>, <tag> </tag>, :```

```

In [ ]:
#!pip install openai==0.28 #for gpt-3.5

In [ ]:
text = f"""
You should express what you want a model to do by \
providing instructions that are as clear and \
specific as you can possibly make them. \
This will guide the model towards the desired output, \
and reduce the chances of receiving irrelevant \
or incorrect responses. Don't confuse writing a \
clear prompt with writing a short prompt. \
In many cases, longer prompts provide more clarity \
and context for the model, which can lead to \
more detailed and relevant outputs.
"""

prompt = f"""
Summarize the text delimited by triple backticks \
into a single sentence.
```{text}```
"""
response = get_completion(prompt)

print(response)

It is important to provide clear and specific instructions to guide a model towards the desired output and reduce the chances of receiving irrelevant or incorrect responses, even if it means writing a longer prompt for more clarity and context.


##DIY: Summarize practice


Use text input from Practice Book  ([Google Doc](https://docs.google.com/document/d/1KB4P6-3_GLn8MTKhrV3OLxlk_9lEDRUPEJJ5ZzhTUOo/edit?usp=sharing)).


In [ ]:
# DIY





### Task 2: Ask for a structured output


- JSON, HTML

In [ ]:
prompt = f"""
Generate a list of three popular movie titles from Korea along \
with their authors and genres.
Provide them in JSON format with the following keys:
movie_id, year, title, author, genre.
"""
response = get_completion(prompt)
print(response)

[
    {
        "movie_id": 1,
        "year": 2003,
        "title": "Oldboy",
        "author": "Park Chan-wook",
        "genre": "Thriller"
    },
    {
        "movie_id": 2,
        "year": 2016,
        "title": "Train to Busan",
        "author": "Yeon Sang-ho",
        "genre": "Horror"
    },
    {
        "movie_id": 3,
        "year": 2019,
        "title": "Parasite",
        "author": "Bong Joon-ho",
        "genre": "Drama"
    }
]


### DIY

E.g., Write a Python script that generates a list of three famous AI researchers along with four variables. Provide them in JSON format. Share your outputs in Practice Book.

In [ ]:
# DIY




### Task 3: Ask the model to check whether conditions are satisfied

2개 이상의 input을 하나의 프롬프트로 처리하기

In [ ]:
text_1 = f"""
Making a cup of tea is easy! First, you need to get some \
water boiling. While that's happening, \
grab a cup and put a tea bag in it. Once the water is \
hot enough, just pour it over the tea bag. \
Let it sit for a bit so the tea can steep. After a \
few minutes, take out the tea bag. If you \
like, you can add some sugar or milk to taste. \
And that's it! You've got yourself a delicious \
cup of tea to enjoy.
"""

In [ ]:
prompt = f"""
You will be provided with text delimited by triple quotes.
If it contains a sequence of instructions, \
re-write those instructions in the following format:

Step 1 - ...
Step 2 - …
…
Step N - …

If the text does not contain a sequence of instructions, \
then simply write \"No steps provided.\"

\"\"\"{text_1}\"\"\"
"""
response = get_completion(prompt)
print("Completion for Text 1:")
print(response)

Completion for Text 1:
Step 1 - Get some water boiling.
Step 2 - Grab a cup and put a tea bag in it.
Step 3 - Pour the hot water over the tea bag.
Step 4 - Let the tea steep for a few minutes.
Step 5 - Remove the tea bag.
Step 6 - Add sugar or milk to taste.


In [ ]:
text_2 = f"""
The sun is shining brightly today, and the birds are \
singing. It's a beautiful day to go for a \
walk in the park. The flowers are blooming, and the \
trees are swaying gently in the breeze. People \
are out and about, enjoying the lovely weather. \
Some are having picnics, while others are playing \
games or simply relaxing on the grass. It's a \
perfect day to spend time outdoors and appreciate the \
beauty of nature.
"""
prompt = f"""
You will be provided with text delimited by triple quotes.
If it contains a sequence of instructions, \
re-write those instructions in the following format:

Step 1 - ...
Step 2 - …
…
Step N - …

If the text does not contain a sequence of instructions, \
then simply write \"No steps provided.\"

\"\"\"{text_2}\"\"\"
"""
response = get_completion(prompt)
print("Completion for Text 2:")
print(response)

Completion for Text 2:
No steps provided.


### DIY: create a sequence of steps.


You can use the ```Condition Check (breaking down to steps)``` text from [Practice Book](https://docs.google.com/document/d/1KB4P6-3_GLn8MTKhrV3OLxlk_9lEDRUPEJJ5ZzhTUOo/edit?tab=t.0).

In [ ]:
text_3 = f""" #DIY text here """
prompt = "# Same prompt used for text_1 and text_2"
response = get_completion(prompt)
print("Completion for Text 3:")
print(response)

### Task 4: "Few-shot" prompting

In [ ]:
prompt = f"""
Your task is to answer in a consistent style.

<child>: Teach me about patience.

<grandparent>: The river that carves the deepest \
valley flows from a modest spring; the \
grandest symphony originates from a single note; \
the most intricate tapestry begins with a solitary thread.

<child>: Teach me about resilience.
"""
response = get_completion(prompt)
print(response)

#### DIY 1: converse reference format into APA style, using few shot prompting

`Jade Smith, Brown, Kim., Trump, D.Analyzing the impact of AI on psychology. 2025. Journal name Cognitive Psychology of Modern Era. Volumn = 11, Issue = 22, Page numbers = 33-45. `

In [ ]:
#DIY





#### **DIY 2**: Convert a text into the desired format by providing a few examples.

직접 input text를 만들어서 zero, one, few-shot prompting 비교해보기

In [ ]:
#DIY





In [ ]:
# DIY




In [ ]:
# DIY




# Principle 2: Give the model time to "think"

### Task 1: Specify the steps required to complete a task

In [ ]:
text = f"""
In a charming village, siblings Jack and Kelly set out on \
a quest to fetch water from a hilltop \
well. As they climbed, singing joyfully, misfortune \
struck—Jack tripped on a stone and tumbled \
down the hill, with Kelly following suit. \
Though slightly battered, the pair returned home to \
comforting embraces. Despite the mishap, \
their adventurous spirits remained undimmed, and they \
continued exploring with delight.
"""
# example 1
prompt_1 = f"""
Perform the following actions:
1 - Summarize the following text delimited by triple \
backticks with 1 sentence.
2 - Translate the summary into Korean.
3 - List each name in the Korean summary.
4 - Output a json object that contains the following \
keys: korean_summary, num_names.

Separate your answers with line breaks.

Text:
```{text}```
"""
response = get_completion(prompt_1)
print("Completion for prompt 1:")
print(response)

Completion for prompt 1:
1 - Jack and Kelly go on a quest to fetch water, but misfortune strikes as Jack trips and tumbles down a hill, with Kelly following suit, yet they return home with comforting embraces, their adventurous spirits undimmed.

2 - 잭과 켈리는 물을 가져오기 위해 여행을 떠나지만 불행이 닥쳐 잭이 넘어지고 언덕을 굴러내리며 켈리가 따라오지만, 그들은 위로받으며 집으로 돌아가고 모험적인 정신은 여전히 유지된다.

3 - 잭, 켈리

4 - 
{
  "korean_summary": "잭과 켈리는 물을 가져오기 위해 여행을 떠나지만 불행이 닥쳐 잭이 넘어지고 언덕을 굴러내리며 켈리가 따라오지만, 그들은 위로받으며 집으로 돌아가고 모험적인 정신은 여전히 유지된다.",
  "num_names": 2
}


### Ask for output in a specified format

In [ ]:
prompt_2 = f"""
Your task is to perform the following actions:
1 - Summarize the following text delimited by
  <> with 1 sentence.
2 - Translate the summary into Korean.
3 - List each name in the Korean summary.
4 - Output a json object that contains the following keys: korean_summary, num_names.

Use the following format:
Text: <text to summarize>
Summary: <summary>
Translation: <summary translation>
Names: <list of names in summary>
Output JSON: <json with summary and num_names>

Text: <{text}>
"""
response = get_completion(prompt_2)
print("\nCompletion for prompt 2:")
print(response)


Completion for prompt 2:
Summary: Jack and Kelly go on a quest to fetch water but encounter misfortune on the way back home.
Translation: 잭과 켈리는 물을 가져오러 가는 여정 중에 불행을 만나지만 집으로 돌아오는 동안에도 모험적인 정신은 사그라들지 않았다.
Names: 잭, 켈리
Output JSON: {"korean_summary": "잭과 켈리는 물을 가져오러 가는 여정 중에 불행을 만나지만 집으로 돌아오는 동안에도 모험적인 정신은 사그라들지 않았다.", "num_names": 2}


#### DIY. Step by Step

Use your own text or text from practice book. Ask the model to 'think' in a specific way to yield better (improved) result from previous ones.

결과만 출력하도록하는 Prompt instruction VS. step-by-step으로 차근차근히 Task를 세분화한 다음에 결과를 출력하도록 하는 Prompt instruction

In [ ]:
# DIY


